# Group 33 — Portuguese student performance

**Original UCI data. Y = G3 (0–20); primary X = studytime (four recorded categories).**

Run **Runtime → Run all** in Google Colab. The original CSV is embedded as a compressed, byte-for-byte copy and checked against its SHA-256 hash. No file upload, Drive mounting, generated observations, API key or extra plotting package is needed.

The notebook recomputes regression, 95% confidence intervals, hypothesis tests, all 30 predictor comparisons and five-fold prediction validation. The last cell creates an offline interactive HTML dashboard for the pitch recording. The original CSV is supplied separately as well.

**Research question:** How do final grades vary across weekly study-time categories? H0: all four means are equal. H1: at least one differs. Alpha = 0.05.

**Primary encoding:** 1: <2 h; 2: 2–5 h; 3: 5–10 h; 4: >10 h. Three dummy coefficients compare categories 2–4 with category 1. The numeric-code SLR is a sensitivity analysis with an equal increment per code step, not per hour.

**Past failures:** a recorded prior outcome and potential risk marker. Largest standalone R² does not mean largest causal effect. The UCI text's coding note is inconsistent with the observed 0–3 values; we preserve those raw codes and use categorical coding primarily.

Source: Cortez, P. (2008). Student Performance [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5TG7T. CC BY 4.0.


In [1]:
# 1. Load the exact original UCI CSV embedded for portability.
from pathlib import Path
import base64, gzip, hashlib, json, html, platform
import numpy as np
import pandas as pd
import scipy
import sklearn

SUBJECT = 'Portuguese'
SUBJECT_CODE = 'POR'
SOURCE_FILENAME = 'student-por.csv'
EXPECTED_SHA256 = 'a7594a11d7771c0efe1a740824e0e833da9c4cad07c39a9766a874575563fb3f'
EMBEDDED_ORIGINAL_GZIP_BASE64 = 'H4sIAAAAAAACA81d264dN3J9z2ecL2iyyO7e2E8CEvklBxiczCCPwRn72FZgSQNdBpl8ffbubt6rilXsrSQQRhY0tkWzyWLVqrVWff3598+f/7h+ffuv6+tvb9fXX3758vb16/XX149fP/z32/VPX7+9fvv+9fr89sv36/v7T8//+fmv1/f3n768vX79/On62/fXL798eP10/fbl9e9vf3z78PHt+vXb91/+sf3q19cPf3y//SuvX7c/6Ov3v23/7ttf/vb64Zfr68/fPvz9w7cPt7/h0/cvX9++/OP6+4fffn/7cv3w6dvbl09v365fPn98/fTtw8/3f+7L2x/XX7+8vW3/6t8+f/7+7frPr3/8fP33+0+/v73+8e336+tfv759+vn2L/zJXH+y15/gn55++tPT9en909Ws16e/3H7505/h9vO7p6u7/Xh6/fYfv3/++Hb7nW9vrz/f/uzbr37+fF/N7RcfP3/bfsfefkzXp3+8fb395qfPxU/7b1b/l7vC7X/m9uP+16fp9tvG3JZgsuUs+XL+/LT93dlyjj86LebX1/13zL6Y7A/vrMjfFgHHYuz16YItxu+L+dd/4RcT/ho2xgg3Jv0NbluE3ZY03xZi76u5/aHGtquJW+Pu696/8O03bgfl7x9+3v6Nx/LScgDbmnYl2c/7cu7/vf7+zxr3tP9kXLaguVrQffXNphyLkXwn8tDY7R/bV2LC1oRD/JxWEj/U/R/KdyQs5svb377fbu+Hz5/azyXdn/3wuLio+L1sZ1H3HaW2B1uL8OS4eKemfWfKlTSXKtzxsIJ0w6u1FPdb+K3MsZj7f8eTmbBvVVyqd/tRw74VszPcYvKtCWfYbFvj76uZ79vRriY7w071kcQHxh8r2U+xxT5Te8Fd8XnCVdecYmZ/4IiAdj+Z+B2v13Tf0O7NCpcddJFn36jw0dy2URN2s3xz3R0eCZFnwmhX5I5tgvzT5bH5Gf10kH+65tmq7pjiq/nt3222x2KiQo+vLjwXevhXgtset32tFHviAfLMI1F+K/KGGfVl3+Pg/b/0HpGXp/2neNnJtWTHJfslccfU5xm2D7Wvy8QADSMvKbokIII0vpqY+YQrZsOK8nv/jORhdZTGLln+9UCSZVQB8n6a9jP1tD5t/1t6bzx1kLD73j3Pxwcz8boXT/vAde8GavETb7J3zNYn6Jl6O+LuNNtEx8PuGQq5j8kTsnqD2o9lRfEQ3xw+PPvjwHl6d0SpGPq17Fhy6NJR3p+w209mKt/Vl/JdLUNR/GLIFoHwxtf1xf7N4pJMtaS5eerJK68I1+jROuJhvGRxPdwlk+XPZjBLvKfyK3WcxaVXWwoa3WW320mw25WPtyuvS5vj/O5IWsUptCQkY3XyfnS4UofKXMkSo01aBT+77WbtP1xYkel+MScsli0dEAkgwW1vvDsC0IR9MX8y8xgIQ+74bOHDbeWPz7KzZyQMVQ9ZwjrYSC1FFVLC6FOx6tolFbEazYVUDweJ/9ijUi0KZ8ucazgZgsgDFG6YwzIOvA5DdqV+3a202omLg6xOdVT48Vgmlg7Nw5CocnMcDkQdn+lFgG8QN0uPjtnqi0GNc2AvPPyvvvCGOEQvzYva4plE2mq0hUaKQHZ7Wh2Gao6gZGw0xFeVwA7Y/sx0sqcesNCmrXXlrLjw28/++FzuyDmGngwEme+ArBKUowB8PR6aJRkihbz039Ty7meFzwXLo9njgzxfKTRuSJYQ6ci2yG6ZonHYZ3uPwJtO83zJU7PjHCWUnlxQmyxmC9IkQv3UdX893NHEAFmmCHheRgQhpy4PbXzmLYYLkek9ee21O1TEw3CAXB6juyAweu3bWp6DgtBl5d0DiyVB5BUTQkEwgNrDAZplDz7gm1TCCw3IQfTl1ItJUXEO2cfMNDTKxZDQi5WVhhls57ek1WWX/RYSL4qz0wPtlPEZttXYCL0Ak7ji4Yf+Usqyed8YF8tmtA02N6i4kSxFg0gdYTn0uW0qmdlu7ru2ZdkPhUNvfNMQc503/hFNKLbgSE/YmlepM5OWGfyRx1vNw2l0EX0c20bAO5iCmrm7Se7oH4S8fq73531d/zRgR/tijH8ue2CbITovoac6M/tjqp4hlyY6tsyg0un0Y8oB15l5xYozRF029ChJE49U/hR9n4k9SYBeNryOVp1siKi06l2VBaXBfYpwnjtQz0mag9CQgzQFkYAO9uAxzExQwkEHTSIrPOVmezr3HGCiKvwWsDLS0OS0z7+LJXW6dhe8zVHmjuJ0X8xAyRcV2ncu7dLU1mgvJcRoOu2glLQ5HbjnjgokMLtQFBb9aqfXIwOKQIY9WFER63RZG2QcvPCxLp1GmenuDVNT91YVMlrHbY7+a/FpbbdVb+MV05MHugW+Oj0KwFUPHu41x8+TqlKp6OleWbuaScjMcWrmQPniWwaN4ZuaCuyDW5U/csf9kiEwNUb0QIuiBn/dCB7s0ak+pIuNBQhY3qWTDEHJRFZk1X2kKnGojMM+1XOv5TICdfL4dOiIQ3mWrXRF5L0CkozMNsZsxhXAcYbs/bSn6bYijmvV8OWaUnL2gq0rfHlTHHIKVQ/JM8zNEiH3RIIRqOOOhvBG2885Wq6LPS4WikSTnkQWH3jjG+VBYAcCg9zzSc+jeIHpBJltf0R5oZXcM/6bdcDgQOeMzLepQz35IRzlglAeqp5Z+mqIY6KO61rc+IvsAKlUCGaA+eaPIzRTFf3chGpxEFIQ/yFmYhbnSJOwmYQ2MMTBg6z5THBce09G7zC7kYTVZcxbUZOeEB01e+T03K6AcZgjCAGhrXkZPtW2TYW6WWOqCl249II02ihJQt1zHa+9rxlLtnuw3QMKQjI5c8dXc/lFgx6qyPTEe7Bip+/iswaiZdRZkuyDaAQpi9TEDjQ2vhqrYkVckxUGaG8lh2muQXMJ3YM51IbhUHa0j17YoPqzWKc1hNmFQ2Qzbs6Ec3Mi3gryDRqkcKdSfsbaHO2CsmuGvx3pFDm9dDYcITMxpPI+uPAI6NfHhwMS8aT+YGhmRh+gwRtWn2fHvPYv3ShdrcXp3viI04ewuGJpNN7PYOUaZ6S8PmIvruQBO5bDQBC42fAjy1v3Fsa+IPRrYS1NkGQd1RFSvhomEoVQaJNE77q5mdFzzVJm5rc3bGUi4oDIxmhx+iT026LPGlhUK3PBtv9oBbVrgGfhGvrbpROE+gf7BInSRxHiLmuZmaoDj0MEWRmr6aUc3JB77McabWUqOwg8g0mimzfHBjmMgbI0BRDHoExa4+28abXGu2TdK2Bg6Gb3vPiQvWcBl3ZJkQ0dRbYpYxDP8SJhPKISszESkeR7r5aNVVmQVkef3zAR5mHzL9bS7tWXq0jtbQGWO7E6qnnjBYm9kORhYn9OVB4+3JciBmdXOx6I7pbw7Bilx0rgTx6f6tJpa7xTeUGoGC++4uGhtO1LtTVTxwgnW4/lCArlT1CKeS8E5Puie0LHc0M4ZOD307NSD8QsaYMhDV1QyjQComlzeMwqZT5MhmFHYGhbEAImrK2yjnh25Aca+kAUVmFsNU/GBFpbC5p4t2SAHajlvIDJeacOUN8zmeoshwvOiSS1gZk453+RaKCkXibc8QnvA+T8FhbOlAAtJ/Kd4EOxVzl7bUqI98vjLPhaTkmwT0F5SxhCs8mwDHKZHtSoeP4QH07DMbReGLIEE5SdGgRH4cJ+86s8NX3cWZl4EZAPV0lIGSUgy3ayT+Zy8xKUCTnL2jrnPlfA5qB25akh+RVrCwqVIrqyL+PVuaP+Q9gtmBMhEAztNuWBsajs9zvGcbVo6mGfu6rRXlca1Qw5NJ0QxMvjJYuhtygDVy40tlLaI0oReSu8XxmbJJm8iNBveZ5qB3rvUAAZQHin9amraJJhtEBY8pc7wMKstBFwMjk2m9W3B2wU9QDZwUW7J0K3GQmKkX5pIkPq9k8eR3ktt2c+JZw7Qf7xSYgFEvtIFWdLDR/YKFC1ONFOcM0evFUu04dZymph5n1a2dRjAKILuYejpPsjnFq9c1EouzZA40A0lqfrLKUg8hTWEZJm7tNqJWJQK+S2dE9y/RsOMynrPRjQMdTNN8eorTZtRpMwkotlHsCGZCij1ePVZ692YWbs8EgCtI2qC3OdO55XJ1Eo/HPVuwQRRvDbH4wwSNCTLMl9nDbiuAiJMUDqikVAQeff8HlYvSx7IAc+U1fdNmZiny0jPMWytKdOCgtyaNdmD4SCRhIzFPmT2QNFA/zJekbg3Zpd06iax0+zi0YqCYK6dGzXiabkuGtA/tkS5XCitqftJYOyySXMLyqBwfZfs+wJ4toiCbh6uEvxAwWWYKNccG9KHtlqcarXQSwBGmNW5pehQ+qPEnDG0FSsPyBl8QMdCbG1pFYbpOR06mrzQJ7wGL39sT9gjaIoZUtk23GqzpMeq+jV7uuA8EzUxwY7xNLQbOvQ3KcZQiZcXKOd74XF5qyoibOdLmkxWoAHO/t6u90z85hTt3sYNajCTOFx3KOrmH7cy9F3I3zM8yJ0v1fhwPBBr1M9YJhlJ9wlhHmPM9vhXdG1DPtR227YK39KNpW7Z/e6F1ZrT5kj6Nh0cOVGNAkHruzQ7iypqhAWMVpJly2A5aUmgT2CXCB0zDSR0khOU5hljtgP0ChWFSeUlOGFfZ6ciNpkaGoTQdwJDQnbAW+zSDP9GEvsCk3OdqcmEC4y98XH2SGVoLIRqMvoeCwCKXuAu+f6NricC36ckbnLmMMTw9bjnFlEky/kjjrhx9RxO9RJJiVrIu9Z4FsV8iB+lAKICYRqCScc63GRxIxc/BeZDI9zZdP1aXdi45q1Rjm41Kkdn1WOmYCxvLselQMGFhIBnjtYRpCWI6ppdAOTxFZMtpgtY7rDpJDJBUKPKAWH2UcPHZ8Sj7Wa7bIMR8YRJ3MfBa+3cG2lRnGgQA20+LuJ5bFJzLma6L10jhJ7+cde/eAWN+ez2xybiFilOlA52Q6KM44FJBTDPGVB371wlSH+zExvA3TwjfoY8T7ZEKQe6DyVlqIv1cHoCFk7k8WnNvKlO7MEFI1b9SAeiI/HTJUdnQGW+gXJhG9Q9gYsc+npTk51xIccJMJ0DkPOdZkHMfkTvHgXLYWJdqCYViySwHQE04DNa7QsH9QpfNYHohAUHqyip4OWTDMNC+EWpaDoMNOz7BCVzz2Zo+WVtFpSAUdpBsSZfn9mdyxe23ccYd3BTIcJ+1ykwBRrlz7Mo8lHdJ70XeTN3x9rDOcPyoiVDia02oRRNwrDRfsYRzl/LBLDqNH2f9PsglzU5avNWbCejtrSQu4hEY6ONQzH50UoMyNqe6d01vKZ98dMLWvhTev4WThG7gWdGUlAFCyurc6ipYeaUy4JnXkPEAeWEaReYS39ANd1kxH4HXWuZ5nyQ0ZA6g+tdrXBBjB3TDNE0g7akOSjddEYdFGDVefskKDu6Fo8ey0fskePi86aUHsX9d4SlKljrLam109Y6HAlViHk+Yh3NTUwadp83TakvpiwmueUDqFLlxnWGZzZUnpVqhFz1Sx2OEzHXGgBeTZTHMGpZIExtYFMMtap8eAVA84wFzSZVWVH7hUs6GkTmwXLy8gQ3WE79yzrIiuADNGc38fjNE21x8/EwAsvjBjuh8yRtJFmTGb2i+xdPeeeWdqyFRo5K/Flq8Gh7mAMhbrcRrtBwJ26yUgtmHUHupkhED0z3fVCy9GELH7NDMfOpE2frE5rM621AYXMqbAoQO+jA1FFN3lP0U1+zBDS4HFqsc7GM4oGwSMaLrwDNGRe1JOEcWx1ekaxN1OGULnjKOFEmEUy3Yl1BRCLGcsumcfwl0U0rV7d9RUuyRGZ9IueszT0YvjC38JKfOeHACrtKQqGuZbOpmVmLec6my2Y5+opQa6nsxwbkdpRNmVKq0srIkTbmcJHfuCepTBkcUXRs9Q6QUR947k57lBgONoPcj6p+pKy8Hyty6WGomajicygw4/oA9l6WBq0qu5MKCO07bRqlZXLeitmpeqLlTed7zXojHLSbxrCHpREK74gHnLpk7K7XFITE9SAHd7SjUWaniK1DqLcYei1DY558BVmSnDxWLkX27YMOMIO9E4lW12fD2pf8orpknxV0cbFckLtCsPMTRuKdtONOGIGfTf6IZR1n3UGOfhJUCELfT/YcYjpUlmmRfCSR0DDkn4zmaBRtlF8qokdKlpkL7imjJB9LB8J7PfzUbHocVMonZ1Ob6hN7XG/EixfVOggEsbAgG9EUP47bAQrti1yX1e9O2iyYCqW43oM3759odUnFPZI0F09DNZTqpR3JEucXY8cfc9HUaMJ6DzIfx6w/neRH75IrfEGJkBr5yDZLCBTveRxwwjVwNwi5Qq9N0NwovrWLA3EDGL31MqjqhpC4tnMAmQ2+8PegdGmzzD9ErW2lINM0aOdxx5DzQ9dVXM+hp0ekwA4Zz32iNh6D0MlzdBtFx3FS5cf7eyBzM7yNEd9UVlynmDP2ZirBxywctu9O6GwI3A7fGLoOKa2xpeH95KNdYwvU64XkpauC32KGVW+0FpwJMcJS7m4zqxMixx4Ib5kGhgmGQXpFHWrrBpc5s63xo+09BQWHWDrc2E7LhwyW6LrUZu3BeGJTdRVZ9jKznDoyfqIbYXNmVl5vzkpjsbOdHDICZ0QV4+keaZKuzNVJjuy2dVORsAkxaYYdiBCbZQyRVtZNE8syPZA5mAj6Ej+sVbiZksLzB43V83GF9PmaejCMJvkA0JH2LDJnbQgOnQ7aJIAmI86IGmeTQB05dTmqQObSEd+w+gQPMjzc1spp1g/I8YbB9RU2EZ+hyzl5WlM0S6f7wTxBHtal3hpcAEgPIMeRuqGCHA5Sr+1DrU99EoXe0TB/YqjM9TYDifzZtrOC1H+lJJQRwbkRW7eiGToTt//dbHOtIRGwU6q15NETfrmLyZ2o23MQxtyzioclnqKdXZsjjlSC1sOwPLMKZbDOMPme5A+FfI0vIzQT0CVc3lsKT1li3a+riz5K99vj6UVK8/FG2Qsde3FCSX0M8UOPCsYQ97uIKHFjZTWCg8F8fTRYTIgRKtGT4RhtgJ/IOnfH294UWmazguu8RUZsamAWECslAZyZV17HskyaSZ0uDwi84mOOXeQWE/C0MmDON+uxv4uuqlkZ+Vih+XeTCkjVg7i6jfP3EAMSnQlQ7JO1geTg+g8mWlzCjThTNJuRf363CffR4txi5kJvX9AWqoA1X1tu9DD+G0pEuO9qOw4EGioKC1JUXuNtAEPvr33MFN87SIAMQMgmNMsAXkSE8bvOh9LqyL0/Gg7MPPKRn9CcfSRuJlV3pY6sqSNLRGHT7Z738uAejERxlpYkDXzAes3XrDkWcmXNvz5bkDUoBIHyntz5dvEj5u3kuNhNh8wN3VOOO/hoamUa38sTPFjKCw10BUlLVCjdzrwyYGzaA5PzHqABDDPWh24mL4ejEUgKJQSeope9kyq6vKqcJG8roqsQzMG2WdBaFKJtB4LftfjFkrNkdOmHQ+oNOI7H18yzqyn/620lC8S2wjD7ibmlZdp6oTDzwUw/LG1+9jqtTrK6PagKjGNek1IU1ZIxYzSJAcGHg2X0e0dA5P158Q/yqTTZ/OrC9PZpVdNDxj2ydmwgVwk4maUhDnNl5M7wuTeKwtxzF+4iW8jrwhv3ZX7vW43755jX1huhMqrE0bOkq39TD3uZ1qiDn2PGjMwLicfAeWITFarRRqH9mw067NlBx4YtMExaPCZ+jU8tba0xXO9BoLwWXMnQ/bMRKG+5dKpw3N0WFxEQIqHbWFIAfjpEbqY9qoyJ7xX5e60E0bO+qu52AXzbMb4ImTJPi72QD4OfSHJ8HwDXrMc7XnuPWPleKwe5KB3W7LF/LAmV3tOpT0lOuaraNCOVczTa2I4+r2p2g4t5VA0lqbFlmW+oJSgl+yCWSueE3jwNgy+EABOGLGkLRW5iy8yyyFQ4cSTstLFiDk3diR1TeNci3rR99xOVKJshdWsqUfQ1dXrwnfKhKo7jZ17muzo64EJ4kf+YTq3fDkoh2xlpTBSEE/jlrOx49M0YNvx6aPt5xCaCah1khB10AWRYumpWRn9wJmJkyaS9lfsRD+jKbTVipc0TdbQW7hQQslOX+GB9nwu2gVWcwOB/VpOyagd2R3HxB9Jl+MB0cdnqgbyui+Njn7MXVpsQJdSIFPI8RZGHagbB6Bxn/MVi1XE9rUCJZUZIFXYOFHMUnd95UqMTkVoBjAXHw3WIn/BMNW7kfDtoIvbVb9rMqGtIfOemjoqXEsa0y63O8iahmsrpF8HwfqxySM+TVuTms2d7R3QQy8bYT8rFBeSRk8QsdOMPii5UxNbBBoFkRV0LZY0DkUx6GPq8EwE8s2e/Nfi44/GG2F6bN4XgJjDdNorN8N1iDTKgQfp3ER0DqRDjzQO/+K5R9nwS9JH7UXik8qObJbnXv4IzJZKBU/poofAb3fEn4ka+j3uujKiuU3Nwp4VgriaoB+MvpjdZVI421Et8n5TyHLcwHJ8fL8mSseOcl04UrQAc0ZPlY+sVmKChqC12/MEVCFP9jBDSIIHwtWDYt0xlSiMNCzyTFkkMRjz1Jf04X2cQe4T/6/3cjFjWB6g3kl54UopDVaFldupKYt5PzkcHNtTfInsYNzIQOA0mR0dIoZOGJA5+oOejeQz/biRLcf+GF5tSDFs2cfxzInRHGE3MKUC6vECjoPj3v2IzckTsCYxnc8QWE+5WKZaglRMtx/L/bAxhskHYcZG8eJ2sbIyYkiAlkZOmpWRX0h8z6W5oIgWVYmbHPs8yLSUqqgTkcGgAHO5bf7CKOIU/NmRFSXPOxSHO6cYBD0w6I6pWJXRphOzjh7jwI6kPFBPE+GpmT+m3VZMm9xwlhiGfLfhpuBBDYuLbBzJa7FW+6XxbwAZF1vjAGqj0r2ezmd6gI9AeW8GphkljlhcToWJ3edBKTHd7ExbnZsZZEb1U+YuZHv1n4qX2VM7oQLQYyCnyfxiLEPLBPkuDb2u/kB4L1JXs4duU89XIgM4Fs5L/52IUXfmnXXxGbFUPFp5t5TzZo8tT7SYMF/t0P3Cqbwws4Bt+0Bdax+9x0ZrcCOFxrSgT17/PAD3RhP9YEEUyx8RJZOkGgs8S5nF+IgjotwjXKfyg8ZL5y5ESYy6dmQhEof4U7BLaOLel4UfaKSaV3GijLZpYI4n9rYwwLirz6gXkTkPBfHQS5BgLdijv2JGvHIm9ihzA4IsrPlw72tq3Xb1JzEh0iQFnzzlD1w/v3cuu/AmPGDQGjWF28Rh7q5KjZ7/jfTcRpPG8Ff5pCxMrB/q6XsKUiSO+XJe9K1UOzqFDqEl2Gw1Hns32KHXH5XBOgejTTEXPO80kwuSdjLswDzVpLV2FSGA+VogZ9Lqxrin0cC+Ic/X29MbJySSgJJdDZMZJd3+sTnSE7ZlNGOlQ36oGmwvnFwI0ZLIYxfqGZniYWRWnaCcXFb3LtDLvfB9JpbVZ4RnJVPoIpTiiVkNmhTWcU9rHO+jEaQPh2XebEPJKy13ItcOfTERl/eNIKZ+D8pHfEAmzN+hZPGVO8NNif9EXmkjG1XYGTzVCXy5/JSLw7hfSo/Ba8bagrsZ0HaEdtEHGYZFdrxWyoZqDLYKgN5UT+bCJcoPfZ+Cr01rO8Y/lxJ/a6t2ezXF1E2X4XN4KGaPssxph2tt+yMG+j2S3/P12w5N2WqWAc5RA/T03imXDd1Lo8bzt9I3c36NQOZ6os3ks3KvVrv//1jVsn+vS7rkmINygc414BM/jIb8gJDZE7gSMKj3RshIxdgZqjO8g95r3JPq8LStW6z99sjPlKc5Bku5RAbT7YcyOsqujz8gePrdP9WFeRfGiFgKaZDNaAh5v5QpqKz83dSUCz5LLiBnIUCvfJGsBgbACh9bFbV9KHnFjXq2pqySgn6qPlaHwxAi6NPsoAtRZf5l6NCYTK0pqqhM1rKd0gy39f98MRD9oNbyNv2FGVI2ouFgvOwhfN4LHmpUA1+UY4XLRyHsy1Q1juoInKXnVjPfXCkUSOmoP95x4aWSzAtyDMeoYw2cDYOdOpGP+Fx19etyXEA49yXhAhXFUoqykR64gwwjE/CEwsJ+dGtO+YjZ43Lbkkzt5JuDzGlMznjqGT1pKg6JD3hMXtJHTcbOjS2VdXl59zx+iqWfqsiKgz/wjGV97yknPNFwT6OkqEAGzV6ah+EZeRiAgCcYIFRo0GML+g4G4MxCO33p+ZUxhPcL7ojarmUVyLr3oMxCc3WCqeQSJCxAeJpIn06e8A4xJ56w9ZCgyUluU28Wda7pBawTMw+zqW1nrDDKprblcBqDr0Y1LxxUeFsz3K6m6NbHZ8zNxOj9nVycZulShmwmBjwxgkFY6l2qLefgaIcQqZcXesxKJUmsx0r51RqACT9BujHvCs5nnX513woQazPtwJRsOADBnYhWzEjU5mBoKQyqPAOymFg4KuWAtmqeuaKIKJ8Ql3nzpMnLRaZ84dnd2KdC+tEioUvqj8ea79JpoFnhkE/Q7EzEujLreBQl8MPjl0VMhnwxSYNNV8MK9ESesCPGyAnoYsET9XRs6ZaUwz03p3j8Eg0/33pZgs8gHCNBlMQ5KcVQZiOeO0QJcK34iXnt8NLl4GEDam23FZ1bW0Es8OYIp7PxxchzCdN1RmuuUcpFZxTdXxVwyUeWzk58pPR5IOAsK7Irq7oE/9gn16OLSyOMmqme/DeSDOAyWndIk8x20lhaMsy9oyoplnQseZCgnDC1EmSOaiKtYV9pCRaA7ZY9DoxPjNr6uMw8+Kjcls7RDS3wqWmuvqdG0p5giHLBLpVxrpQXAbs3Rqy1NEPqogCy2crwguyDAzkPEZXqGrEwJPN/T6OmJuyKsyPCB+yiZB1N06jC+PdSph6WOii0qnyb23DUXIplEG3rDs7F9AX5dO6DfrMwFbgTuS+aoYEc/uAWr9UwFbL1wWhm2ha003eofFV5r4K+UO+1OkKIeNRM0KWZ2tm0Bq8vCnOAc1JUn10pi0XCRVbiksx0JRfcZcSOiYCwG648kOVl2/2wWl9cOISWLhPKXPBXXU6z08pASqytmdVoKmRrGeyKdzijVFM8p69e0NRLXO0ON4V81U7Mx6Tx5d1jp0eSC/IYEXxpWM+ThBFplTc9Zy8Y4gyvY/1Eo8/YXaY8vcQyfGHzQaP0zZMOIzPxf0dmOvM0iloFIwTWFGoqc+zPti8VB1s4KJw7wfIp3TZKX7fAd6kQdJK4FQsrYj6KxBeTs7OprEDyjIssa8RCKq3Tj4vq4No0D2+4vjucmTQ25UYFLNmotjCBb+d5hiaVtj9oRKyPYdCGS7Ykiu97JDEVhh4YAt1s8kqZUdqUUCnERWWFJyWUO9MABvOwSbAdF5hCIEFXPRjUcxYZ7SnzH+LLmfTDbif4/rW4ew64eElzaPA+q82HzSOEZ3bWh3pQt0C7ZHPfNVeFv9bU3kh0VEadUdi4FoNxcPCWr7jGc7JqpuDR5jadk6gPZGUZjtEMiUmrMYe9YkDfLkzIY4ay6CWt6N6kb0WRwYcN8gZa4ebQ+kNgzOetKVL2IQrEoE6PAzgJyARRkv5iKamFIl2n0hybj127oEIq2oacVfvaQZuIwkXDd6gdFX+/QdrPGJEH/wNf+jBZVvhhT9HlO3NqfCYWv2DVuJe0Ngkel36eaTLPWqm43C7IKKy3zZCD8hF+TI4zqalKKnsxdNeSeV9me2u4XjT1ho5q39L/GbxyjmnqF0m/CNi5xSMqlGpbTDRg6amirUy9WVedPSaFjxLFdMUvnb64wkzajhjShUJ4fz8njOO7sKUwg3NtrDMtIBkMKmxMLlZWViAbs2SUvsT+KBZIDuKzwtFVCCxxd8tn/Oc1Nho5klSvmhqf1+OjZ3MmYkcezhcy0LS3fNinJzybuQ8wdr8rLNJIRxXbkdHk4fRYiv3SXimFx4pKNgT4IIaVynHeKag4VjN6O7MKxKc6MtdKoG+wunmX+dGxuUzHVYsRcT2wYyM3Uas5MLYlNWAuYSigPjbIvrTAsYkBWZjVSxtnQq6HEC6BAyg5NHhku74v22wMgEFxm6rZcrDZvaAnhkVpz4PqDU3eEUJx3O7FCOMMjLxPkeW84Od3yFdqSJQdLVbWEGEmltY2dQ5w2UdUqagQKjrBGMCDzLB8gDrDPkJKGJz0TDVjhHQ7bZ0Zaqig7JoEfC4FWdSqE7+CjZ5U61ziJx9+ZVSLMXEcvCcsRGbMz4mjBZ0YyOxj+W2RRuvzQEt8vD1VD1WZEJwNbfsKRvr2DYJwB1Sb1y0LvxbQ0Ld05tCQz8127ZG58HV3L9yATraZHCAO46T7wbm0sYZivLAP5kiWFRDRy745a6+BKB/dZgenacJBaZ0jw3dlPpeV+vjSnwr9dMkHHvMJeo/42CkIAwJX4fq3bWD87RCS35u9C1Pxmg7hWI3qN4+4TbSOKvTJ/I3PRxqXnRcTmNiG47uox52O2VTuTLIVS9DJ+HfKkwy96zbrra6EedJFmGyVFwxkBojttkC2LRdJZ0o21NgMeN+EsQGNxALP+UAuPJG4d6JsF8D0W92vhLtzg8rYrx6t7EjlSWveKZRNWZ31A+Q6UYTAgc5+HZqh13NXLXx5j5UsDKHOSt0OVfrHyPlO4oq5ZNcsjevi9IO8eDXmqpmGjH+QSNyeGShRDlGG4HG99MBgyw+N6jkT9Y13TKhSDnINKa5TD8RV2XOEHBiSbdPSch/b/ck73vrcgfl2qc3scybUheVYS0151R7yPjefWAghpEIu8BEZMS2rbSGz5YWY+S7MSiyZb3Z9ZxXDZwMjinANWAa0+o8IO+1wJpw6Sz9RCs6jiKnFWCus0vGzah4m++ECkL9iXY61Q0sSa/EUc3pDlwxC64UlyWv7QEOT4hzeBzozLhOEyhNM6RH1+xVIsfIhccxACSdBmgiJrpSD+4p1wIWMLd384qb09ZQ/rhQzrhtl0PFVxlINKKee9a67PX2K8f0yEZm1GYsXe71UmcbQYPlCZubqAYzAOAsQg6G6Xk7C/nM+AGXacZOpycZGLQa6VXB9esLgLFuYhowZHmBsUcVifASwLbExnayZwdLdEMcYYjmxYvG4LidUEw7t2MmBbAoT2vtd+SJUKo0WNaN9RLNL14Gu55+wF22VFitpUMJEbc+lgt76nTNJzxVdHhxqf5cUXhNadnU1480TCgPHOWyNx8RCOI0MZKuxGr+XpsdJWsGuJ1y3VINi4uVyNYDQm4dCk/+k9Bcpe9VS85fQ2XTnghCzSybTXkzVwHu+O00DuCMDJQrTk8DkckzuIxiOci7fwGztLpgQuE43jCKLVxNrQ/YcZcmeUwjyl0yoOpPIb+GYRIuaay680qvTcNQ2+Gw2qsphJxonVA0w8ESzJWw80ZHxVtPv2rY5DBqF9d6x3H8AgcbIqp2NiOdesZyc6DENrrTsIs1rjZahbaObbgJXL2z1BfIINGBnafuzmdB6ufXTbUtTbUUI0VooEmR8AyQIfei5QlnIVcz7WiCUPRBmliILlB6+Gtw0Vmo606Iad3HGjcBH9YNDjczWhi/uxNp29TPq8umBuH3FiPOd067EZfMlHPWNLkon1nG/udyARfGVZOoUM6Cud/GJMJQZ/irxV+vN0BLsjImzXE0+WWfukW2FOLxutmzZ0jfb7N0VdW+UwfANM1oJwMPhw+aOeivCyxwFTjSTSWgOSPCYkp3wxLI3QVY/WDVJ0eX3CVeYXpQ4rridTnCYIOGCK/pU8s0IpKgywlOSXWoXqB9La5mxjmnZrVpWD8XghDi62rOIIGAJzCkbo4BNVgNtLy1cwWEDHYscGPRuDIJSR7rVtp4iMlor7VWLfjMfoWRfPpUc+r/9GfrnSd5Tayo7X70JY910q6Gt75QdE0H/OWtEXBgzD1BIo7XDOG302CQmaWOpuZgTDerRoGFIva/hwP8Bgvf3HyRsAQA='
raw_csv = gzip.decompress(base64.b64decode(EMBEDDED_ORIGINAL_GZIP_BASE64))
assert hashlib.sha256(raw_csv).hexdigest() == EXPECTED_SHA256
from io import BytesIO
df = pd.read_csv(BytesIO(raw_csv), sep=';')
assert df.shape == (649, 33)
assert df.isna().sum().sum() == 0
print(f"{SUBJECT}: {len(df)} original records, {len(df.columns)} columns")
print("Source SHA-256:", EXPECTED_SHA256)
print("Zero grades retained:", int((df.G3 == 0).sum()))
print("Versions:", platform.python_version(), "NumPy", np.__version__,
      "pandas", pd.__version__, "SciPy", scipy.__version__, "scikit-learn", sklearn.__version__)

def show_html(content):
    # Colab/Jupyter renders this. A plain Python run still saves the dashboard.
    try:
        from IPython.display import HTML, display
    except ImportError:
        return
    display(HTML(content))


Portuguese: 649 original records, 33 columns
Source SHA-256: a7594a11d7771c0efe1a740824e0e833da9c4cad07c39a9766a874575563fb3f
Zero grades retained: 15
Versions: 3.12.14 NumPy 2.3.5 pandas 2.2.3 SciPy 1.17.0 scikit-learn 1.8.0


## Transparent statistical functions
OLS uses NumPy least squares. Classical t intervals and F tests use SciPy. HC3 provides a heteroskedasticity sensitivity check. The categorical OLS/ANOVA identity and numeric R²/correlation identity are checked independently. All original rows, including zero grades, are retained.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

NUMERIC = {"age", "absences"}
SEED = 42

def design(df, columns, numeric_override=()):
    parts = [np.ones((len(df), 1))]
    names = ["Intercept"]
    for col in columns:
        if col in NUMERIC or col in numeric_override:
            parts.append(df[[col]].to_numpy(dtype=float))
            names.append(col)
        else:
            levels = sorted(df[col].unique())
            for level in levels[1:]:
                parts.append((df[col].to_numpy() == level).astype(float)[:, None])
                names.append(f"{col}[{level}] vs {levels[0]}")
    return np.column_stack(parts), names

def ols(X, y, names):
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)
    n, k = X.shape
    rank = np.linalg.matrix_rank(X)
    assert rank == k, "The design matrix must have full column rank."
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    residual = y - X @ beta
    sse = float(residual @ residual)
    sst = float(((y-y.mean())**2).sum())
    df_resid = n-k
    mse = sse / df_resid
    inv = np.linalg.inv(X.T @ X)
    cov = mse * inv
    se = np.sqrt(np.diag(cov))
    tcrit = stats.t.ppf(.975, df_resid)
    p = 2*stats.t.sf(np.abs(beta/se), df_resid)
    h = np.sum((X @ inv) * X, axis=1)
    hc3 = inv @ (X.T @ ((residual/(1-h))[:, None]**2 * X)) @ inv
    hse = np.sqrt(np.diag(hc3))
    hp = 2*stats.t.sf(np.abs(beta/hse), df_resid)
    q = k-1
    F = ((sst-sse)/q)/mse if q else 0.0
    Fp = stats.f.sf(F, q, df_resid) if q else 1.0
    if q:
        robust_F = float(beta[1:] @ np.linalg.pinv(hc3[1:,1:]) @ beta[1:]/q)
        robust_p = float(stats.f.sf(robust_F, q, df_resid))
    else:
        robust_F, robust_p = 0.0, 1.0
    rows = []
    for j, name in enumerate(names):
        rows.append(dict(term=name, estimate=float(beta[j]), se=float(se[j]),
                         ci_low=float(beta[j]-tcrit*se[j]), ci_high=float(beta[j]+tcrit*se[j]),
                         p=float(p[j]), hc3_se=float(hse[j]), hc3_p=float(hp[j]),
                         hc3_low=float(beta[j]-tcrit*hse[j]), hc3_high=float(beta[j]+tcrit*hse[j])))
    return dict(n=n, k=k, df_resid=df_resid, r2=1-sse/sst,
                adjusted_r2=1-(sse/df_resid)/(sst/(n-1)), rmse=float(np.sqrt(sse/n)),
                residual_sd=float(np.sqrt(mse)), F=float(F), p=float(Fp),
                hc3_F=robust_F, hc3_p=robust_p, coefficients=rows,
                beta=beta.tolist(), fitted=(X@beta).tolist(), residuals=residual.tolist())

def fit_model(df, cols, numeric_override=()):
    X, names = design(df, cols, numeric_override)
    return ols(X, df.G3.to_numpy(), names)

def group_summary(df, predictor):
    result=[]
    for level, sub in df.groupby(predictor, sort=True):
        n=len(sub); mean=float(sub.G3.mean()); sd=float(sub.G3.std(ddof=1))
        error=float(stats.t.ppf(.975,n-1)*sd/np.sqrt(n))
        result.append(dict(level=int(level) if isinstance(level,(int,np.integer)) else str(level),
                           n=n, mean=mean, sd=sd, ci_low=mean-error, ci_high=mean+error))
    return result

def cv_linear(df, cols, numeric_override=()):
    X,_=design(df,cols,numeric_override)
    y=df.G3.to_numpy(dtype=float); pred=np.zeros(len(y)); base=np.zeros(len(y))
    for train,test in KFold(5,shuffle=True,random_state=SEED).split(X):
        pred[test]=X[test]@np.linalg.lstsq(X[train],y[train],rcond=None)[0]
        base[test]=y[train].mean()
    return dict(r2=float(r2_score(y,pred)), rmse=float(np.sqrt(mean_squared_error(y,pred))),
                baseline_rmse=float(np.sqrt(mean_squared_error(y,base))),
                predictions=pred.tolist())

def analyze(df, label, run_ml=True):
    assert df.shape[1]==33 and not df.isna().any().any()
    assert set(df.studytime)=={1,2,3,4} and df.G3.between(0,20).all()
    predictors=[c for c in df.columns if c not in ("G1","G2","G3")]
    main=fit_model(df,["studytime"])
    failure=fit_model(df,["failures"])
    numeric_study=fit_model(df,["studytime"],("studytime",))
    numeric_fail=fit_model(df,["failures"],("failures",))
    famsup=fit_model(df,["famsup"])
    groups=group_summary(df,"studytime")
    fg=group_summary(df,"failures")
    ranking=[]
    for x in predictors:
        m=fit_model(df,[x])
        ranking.append(dict(predictor=x, encoding="numeric" if x in NUMERIC else "categorical",
                            df=m["k"]-1, r2=m["r2"], adjusted_r2=m["adjusted_r2"], p=m["p"]))
    ranking.sort(key=lambda r:r["r2"],reverse=True)
    # Independent identities verify the main fit and coding.
    one_way=stats.f_oneway(*[sub.G3.to_numpy() for _,sub in df.groupby("studytime")])
    assert np.isclose(main["F"],one_way.statistic) and np.isclose(main["p"],one_way.pvalue)
    assert np.isclose(numeric_study["r2"],np.corrcoef(df.studytime,df.G3)[0,1]**2)
    assert np.isclose(numeric_fail["r2"],np.corrcoef(df.failures,df.G3)[0,1]**2)
    assert all(np.isclose(np.mean(np.array(main["fitted"])[df.studytime==g["level"]]),g["mean"]) for g in groups)
    adjusted=fit_model(df,["studytime","failures","school","sex","age","Medu","Fedu","traveltime"])
    X,names=design(df,["studytime","failures","school","sex","age","Medu","Fedu","traveltime"])
    # Partial F: studytime after the stated measured covariates. This is not causal identification.
    reduced=fit_model(df,["failures","school","sex","age","Medu","Fedu","traveltime"])
    sse_full=np.sum(np.array(adjusted["residuals"])**2)
    sse_reduced=np.sum(np.array(reduced["residuals"])**2)
    partial_F=(sse_reduced-sse_full)/3/(sse_full/adjusted["df_resid"])
    adjusted["studytime_partial_F"]=float(partial_F)
    adjusted["studytime_partial_p"]=float(stats.f.sf(partial_F,3,adjusted["df_resid"]))
    positive=fit_model(df[df.G3>0],["studytime"])
    positive_fail=fit_model(df[df.G3>0],["failures"])
    result=dict(subject=label,n=len(df),columns=len(df.columns),missing=int(df.isna().sum().sum()),
                duplicate_rows=int(df.duplicated().sum()),zero_grades=int((df.G3==0).sum()),
                g3_mean=float(df.G3.mean()),g3_sd=float(df.G3.std(ddof=1)),
                study=main,failures=failure,study_numeric=numeric_study,failures_numeric=numeric_fail,
                famsup=famsup,groups=groups,failure_groups=fg,ranking=ranking,adjusted=adjusted,
                cv_study=cv_linear(df,["studytime"]),cv_failures=cv_linear(df,["failures"]),
                cv_study_numeric=cv_linear(df,["studytime"],("studytime",)),
                positive_sensitivity=dict(n=int((df.G3>0).sum()),study_r2=positive["r2"],
                                          study_p=positive["p"],failures_r2=positive_fail["r2"]),
                earlier_grades={x:fit_model(df,[x],(x,))["r2"] for x in ["G1","G2"]})
    if run_ml:
        Xdf=pd.get_dummies(df[predictors],columns=[c for c in predictors if c not in NUMERIC],dtype=float)
        X=Xdf.to_numpy(); y=df.G3.to_numpy(dtype=float)
        mapping={c:[j for j,encoded in enumerate(Xdf.columns) if encoded==c or encoded.startswith(c+"_")] for c in predictors}
        pred=np.zeros(len(y)); importances={c:[] for c in predictors}; fold_metrics=[]
        for fold,(train,test) in enumerate(KFold(5,shuffle=True,random_state=SEED).split(X)):
            model=RandomForestRegressor(n_estimators=150,max_depth=6,min_samples_leaf=5,
                                        max_features=1.0,random_state=SEED+fold,n_jobs=1)
            model.fit(X[train],y[train]); pred[test]=model.predict(X[test])
            score=r2_score(y[test],pred[test]); fold_metrics.append(float(score))
            rng=np.random.default_rng(1000+fold)
            for c,indices in mapping.items():
                drops=[]
                for repeat in range(5):
                    shuffled=X[test].copy(); perm=rng.permutation(len(test))
                    shuffled[:,indices]=X[test][perm][:,indices]
                    drops.append(score-r2_score(y[test],model.predict(shuffled)))
                importances[c].append(float(np.mean(drops)))
        importance=[dict(predictor=c,mean_drop_r2=float(np.mean(v)),fold_sd=float(np.std(v,ddof=1)),folds=v)
                    for c,v in importances.items()]
        importance.sort(key=lambda r:r["mean_drop_r2"],reverse=True)
        result["ml"]=dict(r2=float(r2_score(y,pred)),rmse=float(np.sqrt(mean_squared_error(y,pred))),
                          fold_r2=fold_metrics,importance=importance,predictions=pred.tolist(),
                          settings=dict(trees=150,max_depth=6,min_samples_leaf=5,folds=5,seed=42,permutations=5))
    return result


In [3]:
# 3. Recompute every reported statistic from the original rows.
results = analyze(df, SUBJECT, run_ml=True)
results.update(source_file=SOURCE_FILENAME, sha256=EXPECTED_SHA256)
def pformat(value):
    return f"{value:.2e}" if value < .001 else f"{value:.3f}"
summary = pd.DataFrame([
    {"Model": "Study time: categorical (primary)", "R2_percent": 100*results["study"]["r2"],
     "overall_p": results["study"]["p"], "HC3_joint_p": results["study"]["hc3_p"]},
    {"Model": "Study time: numeric-code SLR", "R2_percent": 100*results["study_numeric"]["r2"],
     "overall_p": results["study_numeric"]["p"], "HC3_joint_p": results["study_numeric"]["hc3_p"]},
    {"Model": "Past failures: categorical", "R2_percent": 100*results["failures"]["r2"],
     "overall_p": results["failures"]["p"], "HC3_joint_p": results["failures"]["hc3_p"]},
    {"Model": "Past failures: numeric code", "R2_percent": 100*results["failures_numeric"]["r2"],
     "overall_p": results["failures_numeric"]["p"], "HC3_joint_p": results["failures_numeric"]["hc3_p"]}
])
print(summary.to_string(index=False, float_format=lambda v: f"{v:.8g}"))
print("Largest standalone R² excluding G1/G2:", results["ranking"][0]["predictor"])
print("Random-forest out-of-fold R²:", results["ml"]["r2"])
print("Random-forest out-of-fold RMSE:", results["ml"]["rmse"])
print("Cross-checks passed: categorical OLS = one-way ANOVA; numeric R² = squared correlation.")
CELL_HTML = summary.to_html(index=False, float_format=lambda v: f"{v:.6g}")
show_html(CELL_HTML)


                            Model  R2_percent     overall_p   HC3_joint_p
Study time: categorical (primary)   6.8765266 5.7057285e-10 2.4083165e-11
     Study time: numeric-code SLR    6.239439 1.0908086e-10 3.0725142e-11
       Past failures: categorical   19.290001 8.6445164e-30 2.0782111e-24
      Past failures: numeric code   15.469712 1.9341533e-25 1.1123686e-19
Largest standalone R² excluding G1/G2: failures
Random-forest out-of-fold R²: 0.2821819100434475
Random-forest out-of-fold RMSE: 2.735036354887116
Cross-checks passed: categorical OLS = one-way ANOVA; numeric R² = squared correlation.


Model,R2_percent,overall_p,HC3_joint_p
Study time: categorical (primary),6.87653,5.70573e-10,2.40832e-11
Study time: numeric-code SLR,6.23944,1.09081e-10,3.07251e-11
Past failures: categorical,19.29,8.64452e-30,2.07821e-24
Past failures: numeric code,15.4697,1.93415e-25,1.11237e-19


## Detailed evidence
Mean-difference CIs use pooled OLS variance. Group-mean CIs use the within-group standard deviation. Neither is an interval for an individual student. The 30-predictor ranking is exploratory, uses in-sample R² and excludes G1/G2. Degrees of freedom differ between models. The optional measured-covariate model is an association analysis, not causal identification.

In [4]:
# 4. Complete numerical evidence, including uncertainty and model diagnostics.
parts = ["<h2>Primary categorical study-time model</h2>",
         pd.DataFrame(results["study"]["coefficients"]).drop(columns=["se","hc3_se"]).to_html(index=False,float_format=lambda v:f"{v:.6g}"),
         "<h3>Actual study-time groups: means and group-specific 95% t CIs</h3>",
         pd.DataFrame(results["groups"]).to_html(index=False,float_format=lambda v:f"{v:.6g}"),
         "<h3>Past failures: group means and intervals</h3>",
         pd.DataFrame(results["failure_groups"]).to_html(index=False,float_format=lambda v:f"{v:.6g}"),
         "<h3>All 30 single-predictor regressions (exploratory)</h3>",
         pd.DataFrame(results["ranking"]).to_html(index=False,float_format=lambda v:f"{v:.6g}"),
         "<h3>Measured-covariate sensitivity</h3>",
         "<p>Studytime + failures + school + sex + age + Medu + Fedu + traveltime. "
         "Partial F-test for all three study-time indicators: p = "+pformat(results["adjusted"]["studytime_partial_p"])+".</p>",
         "<h3>Restricting to positive G3 changes the population</h3>",
         "<pre>"+html.escape(json.dumps(results["positive_sensitivity"],indent=2))+"</pre>"]
CELL_HTML = "".join(parts)
show_html(CELL_HTML)
print("Full result tables generated. Zero-grade sensitivity:", results["positive_sensitivity"])


Full result tables generated. Zero-grade sensitivity: {'n': 634, 'study_r2': 0.06839184270004373, 'study_p': 1.0894029320162583e-09, 'failures_r2': 0.17690408483643538}


<HTML output; rerun this cell in Colab for interactivity.>

## Interactive dashboard for the pitch
Choose **Study time / Past failures**, change **Categorical / Numeric**, highlight a category, and hover on points or means. Open **30-predictor comparison** and **Prediction & validation** for supporting evidence. Highlighting changes emphasis only; all displayed regression statistics keep the full original sample. Horizontal jitter only separates overlapping points and does not change data.

The dashboard can also be opened as the supplied `.html` file in a desktop browser without Python or internet access. If a notebook viewer blocks interactive HTML, run this cell in Colab or download the generated HTML. This is an interactive analysis tool, not a pitch script.

In [5]:
# 5. Create the interactive dashboard and show it inside the notebook.
DASHBOARD_TEMPLATE = '<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Group 33 | Student performance</title>\n<style>\n*{box-sizing:border-box}body{margin:0;background:#f5f7f8;color:#17232d;font:16px/1.5 Arial,Helvetica,sans-serif}main{max-width:1230px;margin:auto;padding:28px 30px 40px}.eyebrow{font-size:12px;letter-spacing:1.4px;text-transform:uppercase;color:#50616b;font-weight:700}h1{font-size:31px;line-height:1.2;margin:8px 0 9px}h2{font-size:20px;margin:0 0 8px}p{margin:8px 0}.sub{color:#4a5a64}.toolbar{display:flex;gap:16px;align-items:end;flex-wrap:wrap;background:white;padding:16px;border:1px solid #d7e0e4;border-radius:10px;margin:22px 0 14px}label{font-size:13px;font-weight:700;display:flex;flex-direction:column;gap:5px}select,button{font:15px Arial;padding:9px 11px;border:1px solid #a3b2bc;border-radius:5px;background:white;color:#17232d;min-height:40px}select:focus,button:focus{outline:3px solid #9fcac8}.tabs{display:flex;gap:6px;flex-wrap:wrap;margin:18px 0}button.active{background:#106d69;color:white;border-color:#106d69}.metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:14px 0}.metric{background:#fff;padding:15px 18px;border:1px solid #d7e0e4;border-radius:8px}.metric .label{font-size:12px;color:#50616b;text-transform:uppercase;letter-spacing:.6px}.metric strong{display:block;font-size:28px;line-height:1.3}.metric small{font-size:12px;color:#50616b}.panel{background:white;border:1px solid #d7e0e4;border-radius:10px;padding:22px;margin-top:14px}.note{font-size:13px;color:#455862}.message{font-size:17px;border-left:4px solid #106d69;background:#edf5f4;padding:12px 16px;margin-top:16px}.chart{width:100%;overflow:auto}svg{display:block;width:100%;height:auto;min-width:650px;font-family:Arial,Helvetica,sans-serif}.legend{display:flex;gap:20px;flex-wrap:wrap;font-size:13px;color:#43545e}.legend span{display:flex;align-items:center;gap:6px}.dot{width:10px;height:10px;border-radius:50%;display:inline-block}.tablewrap{overflow:auto}table{border-collapse:collapse;width:100%;font-size:14px;margin-top:12px}th,td{padding:9px 12px;border-bottom:1px solid #dce4e7;text-align:right}th:first-child,td:first-child{text-align:left}th{font-weight:700}a{color:#106d69}.footer{font-size:12px;color:#52646e;margin-top:24px}.tooltip{position:fixed;display:none;z-index:99;max-width:290px;background:#132e38;color:white;border-radius:6px;padding:10px 13px;font-size:13px;box-shadow:0 3px 16px #0003;pointer-events:none}.hidden{display:none}.key{font-weight:700}.twocol{display:grid;grid-template-columns:1fr 1fr;gap:20px}@media(max-width:760px){main{padding:18px 12px}.metrics{grid-template-columns:repeat(2,1fr)}.twocol{grid-template-columns:1fr}h1{font-size:26px}}\n</style></head><body><main>\n<div class="eyebrow">ACDT Group 33 · Original UCI data</div>\n<h1 id="title"></h1><p class="sub" id="subtitle"></p>\n<div class="tabs" role="navigation" aria-label="Analysis views">\n<button id="tab-explore" class="active" onclick="setView(\'explore\')">Study time &amp; past failures</button>\n<button id="tab-ranking" onclick="setView(\'ranking\')">30-predictor comparison</button>\n<button id="tab-validation" onclick="setView(\'validation\')">Prediction &amp; validation</button>\n</div>\n<section id="explore">\n<div class="toolbar">\n<label>Predictor X<select id="predictor"><option value="studytime">Weekly study time</option><option value="failures">Past class failures</option></select></label>\n<label>Regression encoding<select id="encoding"><option value="categorical">Categorical — primary analysis</option><option value="numeric">Numeric score — sensitivity analysis</option></select></label>\n<label>Highlight recorded category<select id="category"></select></label>\n<button id="reset">Reset view</button>\n</div>\n<div class="metrics" id="metrics"></div>\n<div class="panel"><h2 id="chart-title"></h2><p class="note" id="chart-note"></p><div id="chart" class="chart"></div>\n<div class="legend"><span><i class="dot" style="background:#718691"></i>Observed grade (horizontal jitter only)</span><span><i class="dot" style="background:#17232d"></i>Group mean + 95% t CI</span><span><i class="dot" style="background:#10817b"></i>Fitted regression value</span></div>\n<p class="message" id="finding"></p><p class="note" id="selected"></p></div>\n<div class="panel"><h2>Recorded groups</h2><div class="tablewrap" id="group-table"></div><p class="note">Group CIs describe uncertainty in each group mean; they are not prediction intervals for individual students. All models retain every original row, including G3 = 0.</p></div>\n</section>\n<section id="ranking" class="hidden"><div class="panel"><h2>Which recorded variable has the largest standalone R²?</h2><p>Thirty separate regressions, with G1 and G2 excluded. Each model contains an intercept and one conceptual predictor.</p><p class="note">Age and absences are numeric. Other predictors, including studytime and failures, are categorical. R² is measured on the full sample. Unequal numbers of parameters and selection from the same data limit comparisons.</p><div class="toolbar"><label>Show<select id="rankcount"><option value="10">Top 10 predictors</option><option value="30">All 30 predictors</option></select></label></div><div id="rank-chart" class="chart"></div><p class="message" id="rank-message"></p><div class="tablewrap" id="rank-table"></div></div></section>\n<section id="validation" class="hidden"><div class="panel"><h2>Can the models predict students left out of training?</h2><p>Five-fold cross-validation; fixed shuffled folds, seed 42. Each student\'s prediction is made by a model fitted without that student.</p><div class="tablewrap" id="cv-table"></div><p class="note">CV R² uses pooled out-of-fold predictions and the full-sample mean in its denominator. RMSE is measured in grade points (0–20). The mean-only baseline predicts each test fold using its training-fold mean.</p></div>\n<div class="panel"><h2>Map C evidence: random-forest reliance on recorded variables</h2><p class="note">All 30 predictors, excluding G1/G2. 150 trees, depth ≤ 6, minimum leaf size 5. Grouped dummy columns are permuted together within each held-out fold, five repeats per variable.</p><div id="ml-chart" class="chart"></div><p class="note">Bars show mean reduction in held-out R² after shuffling a predictor. This is model reliance, not explained-variance share or a causal effect. Correlated variables, nonlinearities and sampling variability affect these scores.</p></div>\n</section>\n<p class="footer">Source: Cortez (2008), UCI Student Performance, <a href="https://doi.org/10.24432/C5TG7T" target="_blank" rel="noopener">doi:10.24432/C5TG7T</a>. CC BY 4.0. Data from two Portuguese schools; findings are observational and context-specific. <a id="download-csv" download>Download the exact source CSV</a>.</p>\n</main><div id="tooltip" class="tooltip" role="status"></div>\n<script>\nconst DATA = __PAYLOAD__;\nconst R = DATA.results;\nconst el = id => document.getElementById(id);\nconst pct = v => (100*v).toFixed(2)+\'%\';\nconst num = (v,d=2) => Number(v).toFixed(d);\nconst pv = v => v<.001 ? Number(v).toExponential(2) : Number(v).toFixed(3);\nconst esc = s => String(s).replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst bands = {1:\'< 2 h\',2:\'2–5 h\',3:\'5–10 h\',4:\'> 10 h\'};\nfunction tooltipAttrs(s){return `data-tip="${esc(s)}" tabindex="0"`;}\nfunction modelFor(x,mode){return R[x===\'studytime\'?(mode===\'numeric\'?\'study_numeric\':\'study\'):(mode===\'numeric\'?\'failures_numeric\':\'failures\')];}\nfunction groupsFor(x){return x===\'studytime\'?R.groups:R.failure_groups;}\nfunction gradePlot(x,mode,selected){\n const gs=groupsFor(x),m=modelFor(x,mode),levels=gs.map(g=>g.level),width=1060,height=460,L=65,T=25,W=945,H=340;\n const xp=v=>L+95+(v-levels[0])*(W-190)/3,yp=v=>T+H-v*H/20;\n let s=`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 ${width} ${height}" aria-label="Observed grades, group means and fitted values">`;\n for(let y=0;y<=20;y+=5){s+=`<line x1="${L}" y1="${yp(y)}" x2="${L+W}" y2="${yp(y)}" stroke="#dce4e7"/><text x="${L-12}" y="${yp(y)+5}" text-anchor="end" font-size="14" fill="#425761">${y}</text>`;}\n s+=`<text transform="translate(20,200) rotate(-90)" text-anchor="middle" font-size="15" fill="#263b46">Final grade G3 (0–20)</text>`;\n DATA.rows.forEach((row,i)=>{const g=row[x],j=((i*37%101)/100-.5)*.42,opacity=(selected===\'all\'||Number(selected)===g)?.35:.045;s+=`<circle cx="${xp(g+j)}" cy="${yp(row.G3)}" r="3.2" fill="#718691" opacity="${opacity}" ${tooltipAttrs(\'Original row \'+row.row+\' | \'+(x===\'studytime\'?\'Study-time category \':\'Past-failure code \')+g+\' | G3 \'+row.G3)} />`;});\n gs.forEach((g,i)=>{const X=xp(g.level),opacity=selected===\'all\'||Number(selected)===g.level?1:.18,fit=mode===\'numeric\'?m.beta[0]+m.beta[1]*g.level:m.beta[0]+(i?m.beta[i]:0);s+=`<g opacity="${opacity}"><line x1="${X-11}" y1="${yp(g.ci_low)}" x2="${X-11}" y2="${yp(g.ci_high)}" stroke="#17232d" stroke-width="2.2"/><line x1="${X-19}" y1="${yp(g.ci_low)}" x2="${X-3}" y2="${yp(g.ci_low)}" stroke="#17232d" stroke-width="2.2"/><line x1="${X-19}" y1="${yp(g.ci_high)}" x2="${X-3}" y2="${yp(g.ci_high)}" stroke="#17232d" stroke-width="2.2"/><circle cx="${X-11}" cy="${yp(g.mean)}" r="6" fill="#17232d" ${tooltipAttrs(\'Category \'+g.level+\' | n=\'+g.n+\' | mean \'+num(g.mean)+\' | 95% group CI [\'+num(g.ci_low)+\', \'+num(g.ci_high)+\']\')}/><circle cx="${X+11}" cy="${yp(fit)}" r="6" fill="#10817b" stroke="white" stroke-width="1.5" ${tooltipAttrs(\'Fitted grade: \'+num(fit)+\' | \'+mode+\' model\')}/></g>`;\n s+=`<text x="${X}" y="${T+H+31}" text-anchor="middle" font-size="15" fill="#263b46">${esc(x===\'studytime\'?bands[g.level]:g.level)}</text><text x="${X}" y="${T+H+55}" text-anchor="middle" font-size="12" fill="#62727b">code ${g.level} · n=${g.n}</text>`;});\n if(mode===\'numeric\'){s+=`<line x1="${xp(levels[0])+11}" y1="${yp(m.beta[0]+m.beta[1]*levels[0])}" x2="${xp(levels[3])+11}" y2="${yp(m.beta[0]+m.beta[1]*levels[3])}" stroke="#10817b" stroke-width="2" stroke-dasharray="6 4"/>`;}\n s+=`<text x="${L+W/2}" y="${height-10}" text-anchor="middle" font-size="14" fill="#263b46">${x===\'studytime\'?\'Recorded weekly study-time categories (not exact hours)\':\'Recorded number of past class failures\'}</text></svg>`;return s;\n}\nfunction bars(rows,key,color,label){const width=1050,left=150,right=100,W=width-left-right,h=rows.length*34+48,max=Math.max(...rows.map(r=>r[key]),.01)*1.05;let s=`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 ${width} ${h}" aria-label="${esc(label)}">`;rows.forEach((r,i)=>{const y=12+i*34,fill=r.predictor===\'failures\'?\'#106d69\':color;s+=`<text x="${left-12}" y="${y+16}" text-anchor="end" font-size="14" fill="#263b46">${esc(r.predictor)}</text><rect x="${left}" y="${y}" width="${Math.max(0,r[key])/max*W}" height="23" rx="3" fill="${fill}" ${tooltipAttrs(r.predictor+\' | \'+label+\': \'+num(r[key]*100,2)+\' percentage points\')}/><text x="${left+Math.max(0,r[key])/max*W+9}" y="${y+16}" font-size="13" fill="#263b46">${num(r[key]*100,2)}${key===\'r2\'?\'%\':\' pp\'}</text>`;});return s+\'</svg>\';}\nfunction table(headers,rows){return \'<table><thead><tr>\'+headers.map(h=>\'<th>\'+h+\'</th>\').join(\'\')+\'</tr></thead><tbody>\'+rows.map(row=>\'<tr>\'+row.map(x=>\'<td>\'+x+\'</td>\').join(\'\')+\'</tr>\').join(\'\')+\'</tbody></table>\';}\nfunction refreshCategories(){const x=el(\'predictor\').value;el(\'category\').innerHTML=\'<option value="all">All categories</option>\'+groupsFor(x).map(g=>`<option value="${g.level}">${x===\'studytime\'?\'Category \'+g.level+\' (\'+bands[g.level]+\')\':\'Past-failure code \'+g.level}</option>`).join(\'\');el(\'category\').value=\'all\';}\nfunction renderExplore(){\n const x=el(\'predictor\').value,mode=el(\'encoding\').value,selected=el(\'category\').value,m=modelFor(x,mode),gs=groupsFor(x),name=x===\'studytime\'?\'Study time\':\'Past failures\';\n const metrics=[[\'Students\',R.n,\'Full original sample\'],[\'R²\',pct(m.r2),\'Full-sample model fit\'],[\'Overall p-value\',pv(m.p),\'F(\'+ (m.k-1)+\', \'+m.df_resid+\') = \'+num(m.F)],[\'Residual RMSE\',num(m.rmse),\'Grade points; in-sample\']];\n el(\'metrics\').innerHTML=metrics.map(v=>`<div class="metric"><div class="label">${v[0]}</div><strong>${v[1]}</strong><small>${v[2]}</small></div>`).join(\'\');\n el(\'chart-title\').textContent=name+\' and final \'+(R.subject===\'Portuguese\'?\'Portuguese\':\'mathematics\')+\' grades\';\n el(\'chart-note\').textContent=mode===\'categorical\'?\'Separate fitted mean for each recorded category. Hover over means or points for details.\':\'Numeric-code sensitivity: one constant grade increment per code step. This is not an effect per extra hour.\';\n el(\'chart\').innerHTML=gradePlot(x,mode,selected);\n el(\'finding\').textContent=x===\'studytime\'?(R.subject===\'Portuguese\'?\'Study-time groups differ on average, but study time alone explains only a small share of grade variation.\':\'Study-time groups show no statistically significant overall difference at α = .05 in the full mathematics sample.\'):\'Past failures have the largest standalone R² among the 30 predictors screened without G1/G2. This is an association, not proof of the largest causal effect.\';\n el(\'selected\').textContent=selected===\'all\'?\'Highlighting a category changes emphasis only; model statistics always refer to the full original sample.\':(()=>{const g=gs.find(g=>g.level===Number(selected));return \'Highlighted category \'+g.level+\': \'+g.n+\' students; observed mean \'+num(g.mean)+\'. Full-sample model statistics remain unchanged.\';})();\n el(\'group-table\').innerHTML=table([\'Recorded category\',\'n\',\'Mean G3\',\'95% group-mean CI\'],gs.map(g=>[x===\'studytime\'?g.level+\' (\'+bands[g.level]+\')\':g.level,g.n,num(g.mean),\'[\'+num(g.ci_low)+\', \'+num(g.ci_high)+\']\']));\n}\nfunction renderRanking(){const count=Number(el(\'rankcount\').value),rs=R.ranking.slice(0,count);el(\'rank-chart\').innerHTML=bars(rs,\'r2\',\'#93aaaE\',\'Standalone R²\');el(\'rank-message\').textContent=\'Past failures: \'+pct(R.failures.r2)+\' with categorical coding; \'+pct(R.failures_numeric.r2)+\' with numeric coding. It ranks first among the 30 screened predictors in both subjects.\';el(\'rank-table\').innerHTML=table([\'Predictor\',\'Encoding\',\'Predictor df\',\'R²\',\'Adjusted R²\',\'Overall p\'],rs.map(r=>[r.predictor,r.encoding,r.df,pct(r.r2),pct(r.adjusted_r2),pv(r.p)]));}\nfunction renderValidation(){el(\'cv-table\').innerHTML=table([\'Model\',\'Out-of-fold R²\',\'Out-of-fold RMSE\'],[[\'Mean-only baseline\',\'—\',num(R.cv_study.baseline_rmse)],[\'Study time (categorical)\',pct(R.cv_study.r2),num(R.cv_study.rmse)],[\'Past failures (categorical)\',pct(R.cv_failures.r2),num(R.cv_failures.rmse)],[\'Random forest (30 predictors)\',pct(R.ml.r2),num(R.ml.rmse)]]);el(\'ml-chart\').innerHTML=bars(R.ml.importance.slice(0,8),\'mean_drop_r2\',\'#93aaae\',\'Mean drop in held-out R²\');}\nfunction setView(view){[\'explore\',\'ranking\',\'validation\'].forEach(v=>{el(v).classList.toggle(\'hidden\',view!==v);el(\'tab-\'+v).classList.toggle(\'active\',view===v);});}\nel(\'title\').textContent=\'Study time and \'+(R.subject===\'Portuguese\'?\'Portuguese\':\'mathematics\')+\' grades\';\nel(\'subtitle\').textContent=R.n+\' student records · Y = G3 (0–20) · \'+R.zero_grades+\' zero grades retained · Two schools in Portugal\';\nel(\'predictor\').addEventListener(\'change\',()=>{refreshCategories();renderExplore();});\nel(\'encoding\').addEventListener(\'change\',renderExplore);el(\'category\').addEventListener(\'change\',renderExplore);el(\'rankcount\').addEventListener(\'change\',renderRanking);\nel(\'reset\').addEventListener(\'click\',()=>{el(\'predictor\').value=\'studytime\';el(\'encoding\').value=\'categorical\';refreshCategories();renderExplore();});\ndocument.addEventListener(\'pointerover\',e=>{const target=e.target.closest(\'[data-tip]\');if(target){el(\'tooltip\').textContent=target.dataset.tip;el(\'tooltip\').style.display=\'block\';}});\ndocument.addEventListener(\'pointermove\',e=>{el(\'tooltip\').style.left=Math.min(e.clientX+16,window.innerWidth-300)+\'px\';el(\'tooltip\').style.top=Math.min(e.clientY+16,window.innerHeight-90)+\'px\';});\ndocument.addEventListener(\'pointerout\',e=>{if(e.target.closest(\'[data-tip]\'))el(\'tooltip\').style.display=\'none\';});\ndocument.addEventListener(\'focusin\',e=>{const t=e.target.closest(\'[data-tip]\');if(t){el(\'tooltip\').textContent=t.dataset.tip;el(\'tooltip\').style.display=\'block\';el(\'tooltip\').style.left=\'30px\';el(\'tooltip\').style.top=\'20px\';}});\ndocument.addEventListener(\'focusout\',()=>{el(\'tooltip\').style.display=\'none\';});\nel(\'download-csv\').href=\'data:text/csv;base64,\'+DATA.csv_base64;el(\'download-csv\').download=R.source_file;\nrefreshCategories();renderExplore();renderRanking();renderValidation();\n</script></body></html>\n'
payload = {
    "results": results,
    "rows": df[["studytime", "failures", "G3"]].assign(row=np.arange(1, len(df)+1)).to_dict("records"),
    "csv_base64": base64.b64encode(raw_csv).decode()
}
dashboard_html = DASHBOARD_TEMPLATE.replace("__PAYLOAD__", json.dumps(payload, allow_nan=False))
dashboard_path = Path(f"Group33_{SUBJECT_CODE}_Interactive_Dashboard.html")
dashboard_path.write_text(dashboard_html, encoding="utf-8")
result_path = Path(f"Group33_{SUBJECT_CODE}_Analysis_Results.json")
result_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
csv_path = Path(SOURCE_FILENAME)
if csv_path.exists():
    assert hashlib.sha256(csv_path.read_bytes()).hexdigest() == EXPECTED_SHA256, "An existing CSV differs; it was not overwritten."
else:
    csv_path.write_bytes(raw_csv)
encoded_dashboard = base64.b64encode(dashboard_html.encode()).decode()
CELL_HTML = ("<iframe title='Group 33 interactive analysis' "
             "style='width:100%;height:1600px;border:1px solid #ccd8df;border-radius:8px' "
             "src='data:text/html;base64,"+encoded_dashboard+"'></iframe>")
show_html(CELL_HTML)
print("Saved:", dashboard_path.name, result_path.name, csv_path.name)
print("Interact with predictor, encoding, category and analysis-view controls. No extra packages or web service are needed for the dashboard.")


Saved: Group33_POR_Interactive_Dashboard.html Group33_POR_Analysis_Results.json student-por.csv
Interact with predictor, encoding, category and analysis-view controls. No extra packages or web service are needed for the dashboard.


<HTML output; rerun this cell in Colab for interactivity.>

## Interpretation and reproducibility

- Separate statistical association, prediction and causal hypotheses. Map A uses regressions; Map B proposes mechanisms; Map C uses a random forest and grouped held-out permutation importance.
- Low R² does not measure the amount of confounding. Non-significance does not establish no effect or no need for support.
- Motivation, learning quality and baseline sleep are not measured in these source files.
- Current-period fields such as absences can limit early-prediction claims. G1/G2 exclusion focuses the question on other recorded characteristics.
- The two subject files overlap partly in students; their separate results are not a matched subject comparison.
- Random-forest settings are fixed: 150 trees, depth 6, minimum leaf size 5; five shuffled folds (seed 42), five grouped permutations per predictor in each held-out fold. Importance is mean reduction in held-out R², not a percentage of causal influence. Fold variability and correlated features limit interpretation.
- Source data are historical observations from two Portuguese schools. They are not a newly collected survey or a representative sample of current Korean students.

References: https://doi.org/10.24432/C5TG7T ; https://scikit-learn.org/stable/modules/permutation_importance.html

Optional Colab download (run only if needed):
```python
from google.colab import files
files.download(str(dashboard_path))
```
